# Building Fast Queries on a CSV
## Reading the Dataset
Use the `csv` module to read the `laptops.csv` file and separate the header from the rows:

In [1]:
import csv

In [2]:
with open('laptops.csv', encoding='utf-8') as file:
    read_file = list(csv.reader(file))
    header = read_file[0]
    rows = read_file[1:]

print(header)
for i in range(5):
    print(rows[i])

['Id', 'Company', 'Product', 'TypeName', 'Inches', 'ScreenResolution', 'Cpu', 'Ram', 'Memory', 'Gpu', 'OpSys', 'Weight', 'Price']
['6571244', 'Apple', 'MacBook Pro', 'Ultrabook', '13.3', 'IPS Panel Retina Display 2560x1600', 'Intel Core i5 2.3GHz', '8GB', '128GB SSD', 'Intel Iris Plus Graphics 640', 'macOS', '1.37kg', '1339']
['7287764', 'Apple', 'Macbook Air', 'Ultrabook', '13.3', '1440x900', 'Intel Core i5 1.8GHz', '8GB', '128GB Flash Storage', 'Intel HD Graphics 6000', 'macOS', '1.34kg', '898']
['3362737', 'HP', '250 G6', 'Notebook', '15.6', 'Full HD 1920x1080', 'Intel Core i5 7200U 2.5GHz', '8GB', '256GB SSD', 'Intel HD Graphics 620', 'No OS', '1.86kg', '575']
['9722156', 'Apple', 'MacBook Pro', 'Ultrabook', '15.4', 'IPS Panel Retina Display 2880x1800', 'Intel Core i7 2.7GHz', '16GB', '512GB SSD', 'AMD Radeon Pro 455', 'macOS', '1.83kg', '2537']
['8550527', 'Apple', 'MacBook Pro', 'Ultrabook', '13.3', 'IPS Panel Retina Display 2560x1600', 'Intel Core i5 3.1GHz', '8GB', '256GB SSD',

## Create an Inventory Сlass
Start implementing a class to represent the inventory. It get the name of the CSV file as argument and reads it into self.header and self.rows:

In [3]:
class Inventory():

    def __init__(self, csv_filename):
        with open(csv_filename, encoding='utf-8') as file:
            read_file = list(csv.reader(file))
            self.header = read_file[0]
            self.rows = read_file[1:]
        for row in self.rows:
            row[-1] = int(row[-1])  # Convert the data in the "Price" column from strings to integers

In [4]:
# Let's check our class and print the headers and number of rows in the dataset
inventory = Inventory('laptops.csv')
print(inventory.header)
print(f'Number of rows: {len(inventory.rows)}')

['Id', 'Company', 'Product', 'TypeName', 'Inches', 'ScreenResolution', 'Cpu', 'Ram', 'Memory', 'Gpu', 'OpSys', 'Weight', 'Price']
Number of rows: 1303


## Finding a Laptop From the Id
Implement a `get_laptop_from_id()` method that given a laptop identifier find the row corresponding to that laptop:

In [5]:
class Inventory():

    def __init__(self, csv_filename):
        with open(csv_filename, encoding='utf-8') as file:
            read_file = list(csv.reader(file))
            self.header = read_file[0]
            self.rows = read_file[1:]
        for row in self.rows:
            row[-1] = int(row[-1])

    def get_laptop_from_id(self, laptop_id):  # step 1
        for row in self.rows:                 # step 2
            if row[0] == laptop_id:
                return row
        return None                           # step 3

In [6]:
# let's check our function in action
inventory = Inventory('laptops.csv')
print(inventory.get_laptop_from_id('3362737'))
print(inventory.get_laptop_from_id('3362736'))

['3362737', 'HP', '250 G6', 'Notebook', '15.6', 'Full HD 1920x1080', 'Intel Core i5 7200U 2.5GHz', '8GB', '256GB SSD', 'Intel HD Graphics 620', 'No OS', '1.86kg', 575]
None


## Improving Id Lookups
Improve the time complexity of finding a laptop with a given ID by precomputing a dictionary that maps laptop IDs to strings.</br>
To improve the time complexity of finding a laptop with a given ID:
* I'll create a dictionary `id_to_row` in which I will add the keys - the laptop ID, and the values - the list with data for this laptop
* I'll create a new method called `get_laptop_from_id_fast()`, which will look up the laptop by its id in the dictionary

In [7]:
class Inventory():

    def __init__(self, csv_filename):
        with open(csv_filename, encoding='utf-8') as file:
            read_file = list(csv.reader(file))
            self.header = read_file[0]
            self.rows = read_file[1:]
        for row in self.rows:
            row[-1] = int(row[-1])
        self.id_to_row = {}                        # step 1
        for row in self.rows:                      # step 2
            self.id_to_row[row[0]] = row

    def get_laptop_from_id(self, laptop_id):
        for row in self.rows:
            if row[0] == laptop_id:
                return row
        return None

    def get_laptop_from_id_fast(self, laptop_id):  # step 3
        if laptop_id in self.id_to_row:            # step 4
            return self.id_to_row[laptop_id]
        return None

In [8]:
# let's test the code
inventory = Inventory('laptops.csv')
print(inventory.get_laptop_from_id('3362737'))
print(inventory.get_laptop_from_id('3362736'))

['3362737', 'HP', '250 G6', 'Notebook', '15.6', 'Full HD 1920x1080', 'Intel Core i5 7200U 2.5GHz', '8GB', '256GB SSD', 'Intel HD Graphics 620', 'No OS', '1.86kg', 575]
None


## Comparing the Performance
I'll compare the performance of the methods `get_laptop_from_id`, which performs a search in a list, and `get_laptop_from_id_fast`, which performs a search in pre-prepared data - in a dictionary `id_to_row`.</br>
To measure the execution time of methods, I import the module `time`. To generate pseudo-random values, I import the module `random`.

In [9]:
import time
import random

In [10]:
# I'll create a list with 10,000 values for laptop IDs
ids = [str(random.randint(1_000_000, 9_999_999)) for _ in range(10_000)]

inventory = Inventory('laptops.csv')
total_time_no_dict = 0
for value in ids:
    start = time.time()
    inventory.get_laptop_from_id(value)
    end = time.time()
    total_time_no_dict += end - start

total_time_dict = 0
for value in ids:
    start = time.time()
    inventory.get_laptop_from_id_fast(value)
    end = time.time()
    total_time_dict += end - start

print(f'Total list search execution time: {total_time_no_dict}')
print(f'Total dictionary search execution time: {total_time_dict}')

Total list search execution time: 0.6725225448608398
Total dictionary search execution time: 0.002011537551879883


### Analysis
We got:
* 0.6725225448608398
* 0.002011537551879883

We can see a significant improve in performance. If we divide `0.6725225448608398` by `0.002011537551879883` we see the dictionary-based lookup was approximately `334` times faster than the list-based lookup in this test.

## Two Laptop Promotion
I'll write a method that determines whether we can spend a given amount of money by buying one or two laptops:

In [11]:
class Inventory():

    def __init__(self, csv_filename):
        with open(csv_filename, encoding='utf-8') as file:
            read_file = list(csv.reader(file))
            self.header = read_file[0]
            self.rows = read_file[1:]
        for row in self.rows:
            row[-1] = int(row[-1])
        self.id_to_row = {}                        
        for row in self.rows:                      
            self.id_to_row[row[0]] = row

    def get_laptop_from_id(self, laptop_id):
        for row in self.rows:
            if row[0] == laptop_id:
                return row
        return None

    def get_laptop_from_id_fast(self, laptop_id):  
        if laptop_id in self.id_to_row:
            return self.id_to_row[laptop_id]
        return None

    def check_promotion_dollars(self, dollars):        # step 1
        for row in self.rows:                          # step 2
            if row[-1] == dollars:
                return True
        for row_1 in self.rows:                        # step 3
            for row_2 in self.rows:
                if row_1[-1] + row_2[-1] == dollars:
                    return True
        return False                                   # step 4

In [12]:
# let's test the code
inventory = Inventory('laptops.csv')
print(inventory.check_promotion_dollars(1_000))
print(inventory.check_promotion_dollars(442))

True
False


## Optimizing Laptop Promotion
Since we only care about whether or not there is a solution, we can store all laptops prices in a set when we initialize the inventory. Then we can check in constant time whether there is a laptop with a given price.</br>
Let's implement this to make our code run faster:

In [13]:
class Inventory():

    def __init__(self, csv_filename):
        with open(csv_filename, encoding='utf-8') as file:
            read_file = list(csv.reader(file))
            self.header = read_file[0]
            self.rows = read_file[1:]
        for row in self.rows:
            row[-1] = int(row[-1])
        self.id_to_row = {}                        
        for row in self.rows:                      
            self.id_to_row[row[0]] = row
        self.prices = set()                           # step 1
        for row in self.rows:                         # step 2
            self.prices.add(row[-1])

    def get_laptop_from_id(self, laptop_id):
        for row in self.rows:
            if row[0] == laptop_id:
                return row
        return None

    def get_laptop_from_id_fast(self, laptop_id):  
        if laptop_id in self.id_to_row:
            return self.id_to_row[laptop_id]
        return None

    def check_promotion_dollars(self, dollars):
        for row in self.rows:
            if row[-1] == dollars:
                return True
        for row_1 in self.rows:
            for row_2 in self.rows:
                if row_1[-1] + row_2[-1] == dollars:
                    return True
        return False

    def check_promotion_dollars_fast(self, dollars):  # step 3
        if dollars in self.prices:                    # step 4
            return True
        for price in self.prices:                     # step 5
            if dollars - price in self.prices:
                return True
        return False                                  # step 6

In [14]:
# let's test the code
inventory = Inventory('laptops.csv')
print(inventory.check_promotion_dollars_fast(1_000))
print(inventory.check_promotion_dollars_fast(442))

True
False


## Comparing Promotion Functions
I'll compare the execution speed of the `check_promotion_dollars` methods, which searches for a price in a list, and `check_promotion_dollars_fast`, which checks for the presence of a price in a set `prices`.</br>
For the transparency of the experiment, I'll create a list of prices in the range of values from 100 to 5000:

In [19]:
prices = [random.randint(100, 5_000) for _ in range(100)]

inventory = Inventory('laptops.csv')
total_time_no_set = 0
for price in prices:
    start = time.time()
    inventory.check_promotion_dollars(price)
    end = time.time()
    total_time_no_set += end - start

total_time_set = 0
for price in prices:
    start = time.time()
    inventory.check_promotion_dollars_fast(price)
    end = time.time()
    total_time_set += end - start

print(f'Total list search execution time: {total_time_no_set}')
print(f'Total execution time for checking for presence in the set: {total_time_set}')

Total list search execution time: 0.8225915431976318
Total execution time for checking for presence in the set: 0.00020074844360351562


### Analysis
We got: 
* 0.8225915431976318
* 0.00020074844360351562
  
We can see a significant improve in performance.</br>
If we div 0.82259154319763181 0.00020074844360351562 we see that the new method is above 4 097 times faster for this input size.

## Finding Laptops Within a Budget
With a budget of D, I would like to find all laptops with a price no higher than D. For this task, I'll use `"binary search"` to quickly search a sorted list, but first I need to sort the list of lists:

In [16]:
class Inventory():

    def __init__(self, csv_filename):
        with open(csv_filename, encoding='utf-8') as file:
            read_file = list(csv.reader(file))
            self.header = read_file[0]
            self.rows = read_file[1:]
        for row in self.rows:
            row[-1] = int(row[-1])
        self.id_to_row = {}                        
        for row in self.rows:                      
            self.id_to_row[row[0]] = row
        self.prices = set()                          
        for row in self.rows:                         
            self.prices.add(row[-1])
        self.rows_by_price = sorted(self.rows, key=lambda row: row[-1])  # step 1

    def get_laptop_from_id(self, laptop_id):
        for row in self.rows:
            if row[0] == laptop_id:
                return row
        return None

    def get_laptop_from_id_fast(self, laptop_id):  
        if laptop_id in self.id_to_row:
            return self.id_to_row[laptop_id]
        return None

    def check_promotion_dollars(self, dollars):
        for row in self.rows:
            if row[-1] == dollars:
                return True
        for row_1 in self.rows:
            for row_2 in self.rows:
                if row_1[-1] + row_2[-1] == dollars:
                    return True
        return False

    def check_promotion_dollars_fast(self, dollars):  
        if dollars in self.prices:                    
            return True
        for price in self.prices:                     
            if dollars - price in self.prices:
                return True
        return False

    def find_first_laptop_more_expensive(self, target_price):         # step 2
        range_start = 0
        range_end = len(self.rows_by_price) - 1
        while range_start < range_end:
            range_middle = (range_start + range_end) // 2
            price = self.rows_by_price[range_middle][-1]
            if price > target_price:
                range_end = range_middle
            else:
                range_start = range_middle + 1
        if self.rows_by_price[range_start][-1] <= target_price:
            return -1
        return range_start

    def show_all_laptops_within_budget(self, price):                  # step 3
        first_laptop = self.find_first_laptop_more_expensive(price)
        if first_laptop == 0:
            print('There are no laptops within the specified budget.')
            return []
        elif first_laptop == -1:
            return self.rows_by_price[:5]
        else:
            return self.rows_by_price[:min(first_laptop, 5)]

In [17]:
# let's test the code
inventory = Inventory('laptops.csv')
laptops = inventory.show_all_laptops_within_budget(1_000)
for laptop in laptops:
    print(laptop)

['3564228', 'Acer', 'C740-C9QX (3205U/2GB/32GB/Chrome', 'Netbook', '11.6', '1366x768', 'Intel Celeron Dual Core 3205U 1.5GHz', '2GB', '32GB SSD', 'Intel HD Graphics', 'Chrome OS', '1.3kg', 174]
['7667029', 'Asus', 'Vivobook E200HA', 'Netbook', '11.6', '1366x768', 'Intel Atom x5-Z8350 1.44GHz', '2GB', '32GB Flash Storage', 'Intel HD Graphics 400', 'Windows 10', '0.98kg', 191]
['1478754', 'Vero', 'V131 (X5-Z8350/4GB/32GB/FHD/W10)', 'Notebook', '13.3', 'Full HD 1920x1080', 'Intel Atom X5-Z8350 1.44GHz', '4GB', '32GB Flash Storage', 'Intel HD Graphics 400', 'Windows 10', '1.35kg', 196]
['4366200', 'Asus', 'E402WA-GA010T (E2-6110/2GB/32GB/W10)', 'Notebook', '14', '1366x768', 'AMD E-Series E2-6110 1.5GHz', '2GB', '32GB Flash Storage', 'AMD Radeon R2', 'Windows 10', '1.65kg', 199]
['3840240', 'Acer', 'Chromebook C910-C2ST', 'Notebook', '15.6', '1366x768', 'Intel Celeron Dual Core 3205U 1.5GHz', '2GB', '16GB SSD', 'Intel HD Graphics', 'Chrome OS', '2.19kg', 199]
